# PINNs on Heat equation

## Problem

- Domain: $(x, t) \in \Omega = (0,1) \times (0,1)$
- Parabolic Boundary: 
    - Initial: $\partial_i \Omega = (0,1) \times \{0\}$
    - State Boundary: $\partial_0 \Omega = \{0\} \times (0,1)$ and $\partial_1 \Omega = \{1\} \times (0,1)$
- Equation: 
    $$ u_t = u_{xx} \hbox{ on } \Omega.$$
- Data: 
    - Initial data: $u(x, t) = \phi(x)$ on $\partial_i \Omega$
    - Boundary data: $u(x, t) = 0$ on $\partial_0 \Omega$ and $\partial_1 \Omega$.

In [39]:
# code starts here
import torch

import torch.nn as nn
import matplotlib.pyplot as plt
import numpy as np

# Reproducibility
torch.manual_seed(42)

__Example__
When the initial data $\phi(x) = \sin (\pi x)$, 
the explicit solution shall be
$$ u(t, x) = \sin (\pi x) e^{- \pi^2 t}.$$

__PDE setup for 2nd order__

Let $u: \mathbb R^2 \mapsto \mathbb R$ be smooth function on $\Omega \in \mathbb R^2$. We index coordinate of $\mathbb R^2$ by $0, 1$. This means that $y \in \mathbb R^2$ is $[y_0, y_1]$. The elementary vectors are $e_0 = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$ and $e_1 = \begin{bmatrix} 0 \\ 1 \end{bmatrix}$.

Let $p = D u = \begin{bmatrix} u_0 \\ u_1 \end{bmatrix}$ and
$Q=D^2 u= \begin{bmatrix} u_{00} & u_{01} \\ u_{10} & u_{11} \end{bmatrix}$ be Gradient vector and Hessian matrix.

The heat equation is $u_t - u_{xx} = 0$. If we map $[y_0, y_1]$ to $[t, x]$ in the domain of heat equation, then it can be written as
$$F(D u, D^2 u) = 0, \hbox{ on } \Omega,$$
where 
$$F(p, Q) = p_0 - Q_{11} = e_0 p - e_{1}^\top Q e_1.$$


In [62]:
class pde_2ord:
    def __init__(
        self,
        rec_domain=None,  # Rectangular domain: [[x0_min, x1_min], [x0_max, x1_max]]
        equation=lambda p, Q: p[:, 0:1] - Q[:, 1, 1:2],  # u_x0 - u_x1x1
        initial_condition=lambda x1: torch.sin(np.pi * x1),  # u(x0=0, x1)
        boundary_condition0=lambda x0: torch.zeros_like(x0),  # u(x0, x1=0)
        boundary_condition1=lambda x0: torch.zeros_like(x0),  # u(x0, x1=1)
    ):
        if rec_domain is None:
            rec_domain = [[0.0, 0.0], [1.0, 1.0]]
        self.domain = rec_domain
        self.equation = equation
        self.initial_condition = initial_condition
        self.boundary_condition0 = boundary_condition0
        self.boundary_condition1 = boundary_condition1

In [63]:
def pinns_model(self, hidden_layers=[20, 20], activation=nn.Tanh):
    return nn.Sequential(
        nn.Linear(2, hidden_layers[0]),
        activation(),
        nn.Linear(hidden_layers[0], hidden_layers[1]),
        activation(),
        nn.Linear(hidden_layers[1], 1),
    )

pde_2ord.pinns_model = pinns_model

In [64]:
def loss_function(self, model, x):  # x: (N, 2) tensor of (x0, x1)
    x = x.clone().detach().requires_grad_(True)
    u = model(x)  # (N, 1)

    grad_u = torch.autograd.grad(
        outputs=u,
        inputs=x,
        grad_outputs=torch.ones_like(u),
        create_graph=True,
    )[0]  # (N, 2), [u_x0, u_x1]

    hessian_u = torch.zeros(x.shape[0], 2, 2, device=x.device, dtype=x.dtype)
    for i in range(2):
        d_ui = grad_u[:, i:i+1]
        row_i = torch.autograd.grad(
            outputs=d_ui,
            inputs=x,
            grad_outputs=torch.ones_like(d_ui),
            retain_graph=True,
            create_graph=True,
        )[0]
        hessian_u[:, i, :] = row_i

    residual = self.equation(grad_u, hessian_u)
    return torch.mean(residual**2)

pde_2ord.loss_function = loss_function

In [59]:
def total_loss(self, model, n_collocation, n_initial, n_boundary):
    x0_min, x1_min = self.domain[0]
    x0_max, x1_max = self.domain[1]

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    x0_colloc = torch.rand(n_collocation, device=device, dtype=dtype) * (x0_max - x0_min) + x0_min
    x1_colloc = torch.rand(n_collocation, device=device, dtype=dtype) * (x1_max - x1_min) + x1_min
    collocation_points = torch.stack([x0_colloc, x1_colloc], dim=1)

    x1_initial = torch.rand(n_initial, device=device, dtype=dtype) * (x1_max - x1_min) + x1_min
    x0_initial = torch.full((n_initial,), x0_min, device=device, dtype=dtype)
    initial_points = torch.stack([x0_initial, x1_initial], dim=1)

    x0_boundary = torch.rand(n_boundary, device=device, dtype=dtype) * (x0_max - x0_min) + x0_min
    boundary_points0 = torch.stack([
        x0_boundary,
        torch.full((n_boundary,), x1_min, device=device, dtype=dtype),
    ], dim=1)
    boundary_points1 = torch.stack([
        x0_boundary,
        torch.full((n_boundary,), x1_max, device=device, dtype=dtype),
    ], dim=1)

    pde_loss = self.loss_function(model, collocation_points)

    initial_pred = model(initial_points).squeeze()
    initial_true = self.initial_condition(initial_points[:, 1])
    ic_loss = torch.mean((initial_pred - initial_true) ** 2)

    boundary_pred0 = model(boundary_points0).squeeze()
    boundary_true0 = self.boundary_condition0(boundary_points0[:, 0])
    bc_loss0 = torch.mean((boundary_pred0 - boundary_true0) ** 2)

    boundary_pred1 = model(boundary_points1).squeeze()
    boundary_true1 = self.boundary_condition1(boundary_points1[:, 0])
    bc_loss1 = torch.mean((boundary_pred1 - boundary_true1) ** 2)

    return pde_loss + ic_loss + bc_loss0 + bc_loss1

pde_2ord.total_loss = total_loss

In [60]:
def train_pinns(
    self,
    model,
    n_collocation=1000,
    n_initial=100,
    n_boundary=100,
    epochs=5000,
    lr=1e-3,
):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        optimizer.zero_grad()
        total = self.total_loss(model, n_collocation, n_initial, n_boundary)
        total.backward()
        optimizer.step()
        if epoch % 500 == 0:
            print(f"Epoch {epoch}: Total Loss={total.item():.6f}")

pde_2ord.train_pinns = train_pinns

In [61]:
# Solve the PDE using PINNs
pde = pde_2ord()
model = pde.pinns_model()
pde.train_pinns(model, epochs=1)

Epoch 0: Total Loss=0.400191
